In [1]:
import pandas as pd

base = './student_resource/student_resource/dataset/train/'
cols = ['entity_id', 'business_name', 'business_address', 'country']

def load(name):
    return pd.read_csv(
        base + name,
        sep='\t',
        usecols=cols,
        dtype={'country': 'category'},
        engine='pyarrow'
    )

df1 = load('train_source1.tsv')
df2 = load('train_source2.tsv')
df3 = load('train_source3.tsv')

df1.head(10)

,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India
5,S1-851869949,Custom Wealth Services LLC,"OH, Columbus, 5559 Orville Avenue",US
6,S1-785847572,Consulting Nyasa Nursing Private Limited,"2505, Tower 1, Oakwood, Runwal Greens, Mulund ...",India
7,S1-27541239,Nexus Anchor Rain,"1111 Church Street, Unit 2007, Nashville, TN",US
8,S1-629417405,Moore Bitwise Inc,"337 Oakland Avenue, Michigan City, IN",US
9,S1-22305073,Dermatology Green Medicine,"294 Meadowcreek Drive, Unit Unit 2, Village Of...",US


In [2]:
import re
import regex          # pip install regex  (run once if it errors)
import unicodedata
import pandas as pd

# keeps letters, Hindi matras (marks), numbers and spaces; everything else becomes a space
PUNCT = regex.compile(r'[^\p{L}\p{M}\p{N}\s]+')
HASH_NUM = re.compile(r"#\s*(\d+)")
SPACES = re.compile(r"\s+")

ADDRESS_REPLACEMENTS = {
    # Road
    "road": "rd", "roads": "rd", "street": "st", "avenue": "ave",
    "boulevard": "blvd", "lane": "ln", "drive": "dr", "highway": "hwy",
    "parkway": "pkwy",
    # Building / unit
    "apartment": "apt", "building": "bldg", "suite": "ste",
    "floor": "fl", "unit": "unit",
    # Directions
    "north": "n", "south": "s", "east": "e", "west": "w",
    "northeast": "ne", "northwest": "nw", "southeast": "se", "southwest": "sw",
    # Location words
    "sector": "sec", "district": "dist", "junction": "jn",
    "near": "nr", "opposite": "opp", "market": "mkt",
    # Number
    "number": "no",
}

def normalize_address(address_list):
    normalized_list = []

    for address in address_list:

        # Missing value
        if pd.isna(address):
            normalized_list.append("")
            continue

        address = str(address)
        address = unicodedata.normalize("NFKC", address)
        address = address.lower()
        address = address.replace("&", " and ")
        address = HASH_NUM.sub(r"\1", address)      # #12 / # 12 -> 12
        address = PUNCT.sub(" ", address)           # remove punctuation
        address = SPACES.sub(" ", address).strip()  # remove extra spaces

        # Replace common address words
        normalized_list.append(
            " ".join(ADDRESS_REPLACEMENTS.get(w, w) for w in address.split())
        )

    return normalized_list

test = [
    "1795 Westchester Drive, High Point, NC",
    "KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",
    "राम मार्केटिंग रोड, दिल्ली"
]
print(normalize_address(test))

['1795 westchester dr high point nc', 'kh no 570 13 new delhi w delhi delhi', 'राम मार्केटिंग रोड दिल्ली']


In [ ]:
# name normalization
import re
import regex
import unicodedata
import pandas as pd

PUNCT = regex.compile(r'[^\p{L}\p{M}\p{N}\s]+')   # same as in the address cell
SPACES = re.compile(r"\s+")

LEGAL_SUFFIXES = {
    "pvt", "private", "ltd", "limited", "llp", "opc", "inc", "incorporated",
    "corp", "corporation", "co", "company", "llc", "lp", "plc", "gmbh",
    "ag", "sa", "sarl", "sas", "bv", "nv", "pte", "pty",
}

def normalize_name(name_list):
    normalized_list = []

    for name in name_list:

        # Missing value - handles NaN and None
        if pd.isna(name):
            normalized_list.append("")
            continue

        name = str(name)
        name = unicodedata.normalize("NFKC", name)
        name = name.lower()
        name = PUNCT.sub(" ", name)             # punctuation -> space
        name = SPACES.sub(" ", name).strip()    # remove extra spaces

        # Remove legal suffixes
        normalized_list.append(
            " ".join(w for w in name.split() if w not in LEGAL_SUFFIXES)
        )

    return normalized_list

test = "राम मार्केटिंग प्राइवेट लिमिटेड"
print(normalize_name([test]))



def norm_col(series, fn):
    u = series.dropna().unique()
    m = dict(zip(u, fn(list(u))))
    return series.map(m).fillna("")

['राम मार्केटिंग प्राइवेट लिमिटेड']


In [4]:
df1.head()

,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India


In [6]:
#same preprocessing for df2 and df3
for d in (df1, df2, df3):
    d['normalized_name'] = norm_col(d['business_name'], normalize_name)
    d['normalized_address'] = norm_col(d['business_address'], normalize_address)
    d['country'] = d['country'].astype(str).str.lower().str.strip().astype('category')

df1 = df1[['entity_id', 'country', 'normalized_name', 'normalized_address']]
df2 = df2[['entity_id', 'country', 'normalized_name', 'normalized_address']]
df3 = df3[['entity_id', 'country', 'normalized_name', 'normalized_address']]

# save so you never redo this after a crash
df1.to_parquet('df1_norm.parquet')
df2.to_parquet('df2_norm.parquet')
df3.to_parquet('df3_norm.parquet')

In [8]:
df2.head()

,entity_id,country,normalized_name,normalized_address
0,S2-166376419,india,राम मार्केटिंग प्राइवेट लिमिटेड,kh no 570 13 new delhi w delhi delhi
1,S2-764573417,us,holloway peak seafood,105 elm st morganton nc
2,S2-639257739,india,आदित्य प्रॉपर्टीज एलएलपी,g 3 571 gulmohar colony bhopal madhya pradesh
3,S2-163963287,us,summit,greensboro nc 19 1 2 stardust trail
4,S2-49942811,us,delta tetlecommunication,914 pierpont ave cleveland oh


In [9]:
df3.head()

,entity_id,country,normalized_name,normalized_address
0,S3-202863386,us,wilfordhancock com,mack rd haltom city texas
1,S3-859268022,india,international south consultants,
2,S3-22467283,us,moncada léarning center,5780 fawn ct fort worth texas
3,S3-671162755,us,moyna s coffee,1 ivanhoe ave po box 6009 cincinnati ohio
4,S3-960981775,india,efs print ventures,door no 183 41st cross 22nd main 9th block jay...


In [ ]:
print(len(df1))
print(len(df2))
print(len(df3))



2206821
5034616
5285603


In [12]:
import os
os.makedirs('pairs', exist_ok=True)

def block_pairs(left, right, key, max_block=200):
    l = left.loc[left[key].ne(''), ['entity_id', 'country', key]].rename(columns={'entity_id': 'id1'})
    r = right.loc[right[key].ne(''), ['entity_id', 'country', key]].rename(columns={'entity_id': 'id2'})
    sizes = r.groupby(['country', key], observed=True)['id2'].transform('size')
    r = r[sizes <= max_block]
    return l.merge(r, on=['country', key])[['id1', 'id2']]

chunk = 250_000
for i, start in enumerate(range(0, len(df1), chunk)):
    part = df1.iloc[start:start + chunk]
    out = pd.concat([
        block_pairs(part, df2, 'normalized_name'),
        block_pairs(part, df3, 'normalized_name'),
        block_pairs(part, df2, 'normalized_address'),
        block_pairs(part, df3, 'normalized_address'),
    ]).drop_duplicates()
    out.to_parquet(f'pairs/part_{i}.parquet')
    print(i, len(out))

pairs = pd.read_parquet('pairs')
print(pairs.groupby('id1').size().describe())

0 5108710
1 5104316
2 5107759
3 5071371
4 5124517
5 5109896
6 5153536
7 5083863
8 4239365
count    1.999316e+06
mean     2.255938e+01
std      4.382050e+01
min      1.000000e+00
25%      2.000000e+00
50%      4.000000e+00
75%      1.600000e+01
max      4.010000e+02
dtype: float64


In [13]:
# ==========================================
# STEP 8: Add address-number candidates
# ONLY for rows having no candidates
# ==========================================

pairs = pd.read_parquet('pairs')

empty = df1[~df1['entity_id'].isin(pairs['id1'].unique())]
print("Rows with 0 candidates:", len(empty))

def num_table(df, id_name):
    t = df[['entity_id', 'country']].rename(columns={'entity_id': id_name})
    t['num'] = df['normalized_address'].str.findall(r'\d+')
    t = t.explode('num')
    return t[t['num'].notna() & t['num'].ne('')]

left = num_table(empty, 'id1')
needed = set(left['num'])

rights = []
for d in (df2, df3):
    r = num_table(d, 'id2')
    rights.append(r[r['num'].isin(needed)])   # keep only numbers we need
r = pd.concat(rights)

# Prevent candidate explosion (drop very common numbers)
r = r[r.groupby(['country', 'num'], observed=True)['id2'].transform('size') <= 100]

num_pairs = (
    left.merge(r, on=['country', 'num'])[['id1', 'id2']]
    .drop_duplicates()
    .groupby('id1').head(100)
)

pairs = pd.concat([pairs, num_pairs], ignore_index=True)


# ==========================================
# STEP 9: Candidate statistics
# ==========================================

candidate_counts = pairs.groupby('id1').size()

print("\nUpdated Candidate Statistics:")
print(candidate_counts.describe())

print("\nRows still having 0 candidates:",
      len(df1) - candidate_counts.shape[0])

Rows with 0 candidates: 207505

Updated Candidate Statistics:
count    2.017537e+06
mean     2.274353e+01
std      4.376065e+01
min      1.000000e+00
25%      2.000000e+00
50%      4.000000e+00
75%      1.700000e+01
max      4.010000e+02
dtype: float64

Rows still having 0 candidates: 189284


In [14]:
# Check blocking recall
ground_truth = pd.read_csv('./student_resource/student_resource/dataset/train/train_ground_truth.tsv', sep='\t')

# one row per (source1 id, true match id)
gt = ground_truth[['source1_entity_id', 'matched_entity_ids']].copy()
gt['id2'] = gt['matched_entity_ids'].astype(str).str.split(',')
gt = gt.explode('id2')
gt['id2'] = gt['id2'].str.strip()

# make dtypes match pairs (ids may be int or str)
gt['id2'] = gt['id2'].astype(pairs['id2'].dtype)
gt['source1_entity_id'] = gt['source1_entity_id'].astype(pairs['id1'].dtype)

# which true pairs were found by blocking
hit = gt.merge(
    pairs,
    left_on=['source1_entity_id', 'id2'],
    right_on=['id1', 'id2'],
    how='inner'
)

found_ids = hit['source1_entity_id'].unique()
total_ids = gt['source1_entity_id'].unique()

blocking_recall = len(found_ids) / len(total_ids)

print("Blocking Recall:", blocking_recall)
print("Blocking Recall %:", blocking_recall * 100)

Blocking Recall: 0.7927303573783284
Blocking Recall %: 79.27303573783284


In [15]:
import os
import pandas as pd

# ==========================================
# FUZZY BLOCKING + NEW RECALL (all in one)
# Needs: df1, df2, df3 (normalized), pairs (from Step 8/9)
# ==========================================

# ---------- 1. Ground truth ----------
gt = pd.read_csv(
    './student_resource/student_resource/dataset/train/train_ground_truth.tsv',
    sep='\t'
)[['source1_entity_id', 'matched_entity_ids']]
gt['id2'] = gt['matched_entity_ids'].astype(str).str.split(',')
gt = gt.explode('id2')
gt['id2'] = gt['id2'].str.strip().astype(pairs['id2'].dtype)
gt['source1_entity_id'] = gt['source1_entity_id'].astype(pairs['id1'].dtype)
total_ids = set(gt['source1_entity_id'].unique())

# ---------- 2. Recall from the existing pairs ----------
hit_old = gt.merge(
    pairs, left_on=['source1_entity_id', 'id2'], right_on=['id1', 'id2'], how='inner'
)
found_ids = set(hit_old['source1_entity_id'].unique())
print("Recall before fuzzy: %.2f%%" % (100 * len(found_ids) / len(total_ids)))
del hit_old

# ---------- 3. Fuzzy keys (prefix of name, prefix of address) ----------
KEYS = {
    'name_prefix': ('normalized_name', 8),
    'addr_prefix': ('normalized_address', 12),
}

def make_keys(d, id_name):
    t = d[['entity_id', 'country']].rename(columns={'entity_id': id_name}).copy()
    for k, (col, n) in KEYS.items():
        t[k] = d[col].str[:n]
    return t

right_all = pd.concat([make_keys(df2, 'id2'), make_keys(df3, 'id2')], ignore_index=True)

# Build the right-side tables ONCE (drop very common prefixes)
MAX_BLOCK = 50
right_tables = {}
for k in KEYS:
    r = right_all.loc[right_all[k].ne(''), ['id2', 'country', k]]
    sizes = r.groupby(['country', k], observed=True)['id2'].transform('size')
    right_tables[k] = r[sizes <= MAX_BLOCK]
del right_all

# ---------- 4. Chunked fuzzy matching, saved to disk ----------
os.makedirs('pairs_fuzzy', exist_ok=True)
for f in os.listdir('pairs_fuzzy'):          # clear old runs
    os.remove(os.path.join('pairs_fuzzy', f))

left_all = make_keys(df1, 'id1')
chunk = 100_000
fuzzy_total = 0

for i, start in enumerate(range(0, len(left_all), chunk)):
    part = left_all.iloc[start:start + chunk]

    outs = []
    for k in KEYS:
        l = part.loc[part[k].ne(''), ['id1', 'country', k]]
        outs.append(l.merge(right_tables[k], on=['country', k])[['id1', 'id2']])

    out = pd.concat(outs).drop_duplicates().groupby('id1').head(100)
    out.to_parquet(f'pairs_fuzzy/part_{i}.parquet')
    fuzzy_total += len(out)

    # update recall with this chunk
    out2 = out.copy()
    out2['id1'] = out2['id1'].astype(pairs['id1'].dtype)
    out2['id2'] = out2['id2'].astype(pairs['id2'].dtype)
    hit = gt.merge(
        out2, left_on=['source1_entity_id', 'id2'], right_on=['id1', 'id2'], how='inner'
    )
    found_ids |= set(hit['source1_entity_id'].unique())
    print(i, len(out))

# ---------- 5. Final recall ----------
recall = len(found_ids) / len(total_ids)
print("\nFuzzy pairs saved:", fuzzy_total)
print("Blocking Recall AFTER fuzzy: %.2f%%" % (recall * 100))

# ---------- 6. Sample of matches still missed (to see what's left) ----------
missed = gt[~gt['source1_entity_id'].isin(found_ids)].head(10)
a = missed.merge(df1, left_on='source1_entity_id', right_on='entity_id')
b = pd.concat([df2, df3]).rename(columns=lambda c: c + '_true')
view = a.merge(b, left_on='id2', right_on='entity_id_true')
pd.set_option('display.max_colwidth', 45)
view[['normalized_name', 'normalized_name_true',
      'normalized_address', 'normalized_address_true',
      'country', 'country_true']]

Recall before fuzzy: 79.27%
0 840514
1 846269
2 849110
3 848322
4 845475
5 845993
6 844147
7 848094
8 849951
9 840782
10 843707
11 844889
12 839211
13 848307
14 845398
15 843301
16 848258
17 846739
18 854121
19 845773
20 846624
21 849209
22 57608

Fuzzy pairs saved: 18671802
Blocking Recall AFTER fuzzy: 88.05%


,normalized_name,normalized_name_true,normalized_address,normalized_address_true,country,country_true
0,ss food,एसएस फूड प्राइवेट लिमिटेड,af 684 nandgram nr mother india public sc...,af 0684 nandgram nr mother india public s...,india,india
1,ss food,एसएस फूड प्राइवेट लिमिटेड,af 684 nandgram nr mother india public sc...,af 0684 uttar pradesh ghaziabad 9487203,india,india
2,ss food,एसएस फूड प्राइवेट लिमिटेड,af 684 nandgram nr mother india public sc...,af 684 ghaziabad up,india,india
3,red consultants,red center,c o tapas kumar betal bhogpur purba medin...,,india,india
4,physical therapy clinic,physlcal therapy clinic,5208 beacon falls dr fl 1 columbia mo,beacon falls dr columbia mo,us,us
5,physical therapy clinic,physical clinic therapy,5208 beacon falls dr fl 1 columbia mo,,us,us
6,physical therapy clinic,physical therapy clinic,5208 beacon falls dr fl 1 columbia mo,005208 beacon falls dr fl 1 columbia miss...,us,us
7,physical therapy clinic,physical therapy clinic,5208 beacon falls dr fl 1 columbia mo,005208 beacon falls dr fl 1 columbia miss...,us,us


In [16]:
import os
import pandas as pd

# ==========================================
# FUZZY BLOCKING ROUND 2 + FINAL RECALL
# Needs: df1, df2, df3 (normalized), norm_col, pairs (Step 8/9),
#        folder pairs_fuzzy/ (from the previous cell)
# ==========================================

# ---------- 0. Save Step-8 pairs once, so a crash can't lose them ----------
if not os.path.exists('pairs_all.parquet'):
    pairs.to_parquet('pairs_all.parquet')

# ---------- 1. Ground truth ----------
gt = pd.read_csv(
    './student_resource/student_resource/dataset/train/train_ground_truth.tsv',
    sep='\t'
)[['source1_entity_id', 'matched_entity_ids']]
gt['id2'] = gt['matched_entity_ids'].astype(str).str.split(',')
gt = gt.explode('id2')
gt['id2'] = gt['id2'].str.strip().astype(str)
gt['source1_entity_id'] = gt['source1_entity_id'].astype(str)
total_ids = set(gt['source1_entity_id'].unique())

def found_from(p):
    h = gt.merge(p, left_on=['source1_entity_id', 'id2'],
                 right_on=['id1', 'id2'], how='inner')
    return set(h['source1_entity_id'].unique())

# ---------- 2. Recall so far (pairs + pairs_fuzzy) ----------
found_ids = found_from(pairs)
for f in sorted(os.listdir('pairs_fuzzy')):
    found_ids |= found_from(pd.read_parquet(f'pairs_fuzzy/{f}'))
print("Recall before round 2: %.2f%%" % (100 * len(found_ids) / len(total_ids)))

# ---------- 3. Key builders (hashed, so memory stays small) ----------
cats = set()
for d in (df1, df2, df3):
    cats |= set(d['country'].astype(str).unique())
cmap = {c: i for i, c in enumerate(sorted(cats))}

KEYS = ['addr_clean_prefix', 'sortname_num', 'num_tok']

def key_strings(d):
    # remove leading zeros from numbers: "0684" -> "684", "005208" -> "5208"
    addr = d['normalized_address'].str.replace(r'\b0+(?=\d)', '', regex=True)
    name = d['normalized_name']
    first_num = addr.str.extract(r'(\d+)', expand=False).fillna('')
    # name words sorted: "physical clinic therapy" == "physical therapy clinic"
    sname = norm_col(name, lambda lst: [' '.join(sorted(s.split())) for s in lst])
    # number + next word: "684 nandgram"
    ex = addr.str.extract(r'\b(\d+)\s+([^\W\d_]+)', expand=True)
    return {
        'addr_clean_prefix': addr.str[:12],
        'sortname_num': (sname + '|' + first_num).where(sname.ne('') & first_num.ne(''), ''),
        'num_tok': (ex[0] + ' ' + ex[1]).fillna(''),
    }

def key_frames(d, id_name):
    ks = key_strings(d)
    ctry = d['country'].astype(str).map(cmap).astype('int16').values
    ids = d['entity_id'].values
    out = {}
    for k, s in ks.items():
        mask = s.ne('').values
        f = pd.DataFrame({id_name: ids[mask], 'country': ctry[mask]})
        f[k] = pd.util.hash_pandas_object(s[mask], index=False).values
        out[k] = f
    return out

# ---------- 4. Right side (S2 + S3) tables, built once ----------
MAX_BLOCK = 50
parts = {k: [] for k in KEYS}
for d in (df2, df3):
    for k, f in key_frames(d, 'id2').items():
        parts[k].append(f)

right_tables = {}
for k in KEYS:
    r = pd.concat(parts[k], ignore_index=True)
    sizes = r.groupby(['country', k])['id2'].transform('size')
    right_tables[k] = r[sizes <= MAX_BLOCK]
del parts, r, sizes

# ---------- 5. Chunked matching for df1, saved to disk ----------
os.makedirs('pairs_fuzzy2', exist_ok=True)
for f in os.listdir('pairs_fuzzy2'):
    os.remove(os.path.join('pairs_fuzzy2', f))

chunk = 100_000
total_new = 0
for i, start in enumerate(range(0, len(df1), chunk)):
    part = df1.iloc[start:start + chunk]
    left = key_frames(part, 'id1')

    outs = [
        left[k].merge(right_tables[k], on=['country', k])[['id1', 'id2']]
        for k in KEYS
    ]
    out = pd.concat(outs).drop_duplicates().groupby('id1').head(100)
    out.to_parquet(f'pairs_fuzzy2/part_{i}.parquet')
    total_new += len(out)
    found_ids |= found_from(out)
    print(i, len(out))

# ---------- 6. Final recall ----------
print("\nRound 2 pairs saved:", total_new)
print("Blocking Recall AFTER round 2: %.2f%%" % (100 * len(found_ids) / len(total_ids)))

# ---------- 7. Sample of matches still missed ----------
missed = gt[~gt['source1_entity_id'].isin(found_ids)].sample(10, random_state=1)
a = missed.merge(df1, left_on='source1_entity_id', right_on='entity_id')
b = pd.concat([df2, df3]).rename(columns=lambda c: c + '_true')
view = a.merge(b, left_on='id2', right_on='entity_id_true')
pd.set_option('display.max_colwidth', 45)
view[['normalized_name', 'normalized_name_true',
      'normalized_address', 'normalized_address_true',
      'country', 'country_true']]

Recall before round 2: 88.05%
0 704994
1 714125
2 715683
3 708732
4 706091
5 706548
6 707456
7 708039
8 703932
9 708079
10 706412
11 707907
12 705862
13 706783
14 708909
15 705643
16 707448
17 714137
18 711050
19 706532
20 709459
21 713800
22 49425

Round 2 pairs saved: 15637046
Blocking Recall AFTER round 2: 90.89%


,normalized_name,normalized_name_true,normalized_address,normalized_address_true,country,country_true
0,lakshmi consultancy,लक्ष्मी कंसल्टेंसी,plot no223 4 kh no 81 25 jain col tiranth...,00223 4 kh no 81 25 jain col tiranthakar ...,india,india
1,all constructions,ఆల్ కన్ స్ట్రక్షన్స్ ప్రైవేట్ లిమిటెడ్,telangana k v rangareddy rangareddy plot ...,door no 8 b 1 and 8 b 2 e city maheshwara...,india,india
2,future future developers,ஃபியூச்சர் ஃபியூச்சர் டெவலப்பர்ஸ் பிரைவேட...,9a c1 basement kings square bldg 11th cro...,9a c1 trichy tiruchirappalli tamil nadu,india,india


In [17]:
import os
import pandas as pd

# ==========================================
# MERGE ALL PAIRS -> ONE DEDUPED FILE (integer ids)
# Reads: pairs_all.parquet, pairs_fuzzy/, pairs_fuzzy2/
# Writes: pairs_final.parquet
# ==========================================

def to_int_id(s):
    # "S1-925783039" -> 1 * 10^10 + 925783039  (unique per source + number)
    s = s.astype(str)
    return s.str[1:2].astype('int64') * 10**10 + s.str[3:].astype('int64')

def load_pairs(path):
    p = pd.read_parquet(path)
    p['id1'] = to_int_id(p['id1'])
    p['id2'] = to_int_id(p['id2'])
    return p

# quick format check on one file (stops early if ids aren't like "S1-123")
test = pd.read_parquet('pairs_fuzzy2/part_0.parquet').head(5)
print(test)
print(to_int_id(test['id1']).tolist(), to_int_id(test['id2']).tolist())

frames = [load_pairs('pairs_all.parquet')]
for folder in ('pairs_fuzzy', 'pairs_fuzzy2'):
    for f in sorted(os.listdir(folder)):
        frames.append(load_pairs(f'{folder}/{f}'))
    print(folder, "loaded")

all_pairs = pd.concat(frames, ignore_index=True)
del frames
print("Total before dedupe:", len(all_pairs))

all_pairs = all_pairs.drop_duplicates().sort_values(['id1', 'id2']).reset_index(drop=True)
print("Total after dedupe:", len(all_pairs))

all_pairs.to_parquet('pairs_final.parquet')

per_row = all_pairs.groupby('id1').size()
print(per_row.describe())
print("S1 rows with at least one candidate:", per_row.shape[0], "of", len(df1))

            id1           id2
0  S1-925783039  S2-157377754
1  S1-925783039  S2-517291332
2  S1-925783039  S3-997698194
3  S1-773889195  S3-202893869
4  S1-773889195  S3-622232873
[10925783039, 10925783039, 10925783039, 10773889195, 10773889195] [20157377754, 20517291332, 30997698194, 30202893869, 30622232873]
pairs_fuzzy loaded
pairs_fuzzy2 loaded
Total before dedupe: 80194759
Total after dedupe: 66078355
count    2.182670e+06
mean     3.027409e+01
std      4.303970e+01
min      1.000000e+00
25%      5.000000e+00
50%      1.300000e+01
75%      4.200000e+01
max      4.430000e+02
dtype: float64
S1 rows with at least one candidate: 2182670 of 2206821


In [18]:
import os
import pandas as pd

# ==========================================
# BLOCKING ROUND 3 (typo / zero-padding / word-order keys)
# Reads : pairs_final.parquet, ground truth, df1, df2, df3
# Writes: pairs_fuzzy3/ (new pairs), pairs_final_v2.parquet (everything merged)
# All ids are integers: "S2-157377754" -> 20157377754
# ==========================================

# ---------- 0. Helpers ----------
def to_int_id(s):
    s = s.astype(str)
    return s.str[1:2].astype('int64') * 10**10 + s.str[3:].astype('int64')

def to_str_id(i):
    return 'S' + (i // 10**10).astype(str) + '-' + (i % 10**10).astype(str)

# ---------- 1. Ground truth (integer ids) ----------
gt = pd.read_csv(
    './student_resource/student_resource/dataset/train/train_ground_truth.tsv',
    sep='\t'
)[['source1_entity_id', 'matched_entity_ids']]
gt['id2'] = gt['matched_entity_ids'].astype(str).str.split(',')
gt = gt.explode('id2')
gt['id2'] = gt['id2'].str.strip()
valid = (gt['id2'].str.match(r'^S\d-\d+$')
         & gt['source1_entity_id'].astype(str).str.match(r'^S\d-\d+$'))
print("Ground-truth rows with no valid match id (ignored):", (~valid).sum())
gt = gt[valid].copy()
gt['id1'] = to_int_id(gt['source1_entity_id'])
gt['id2'] = to_int_id(gt['id2'])
gt = gt[['id1', 'id2']]
total_ids = set(gt['id1'].unique())

def found_from(p):
    return set(gt.merge(p, on=['id1', 'id2'])['id1'].unique())

# ---------- 2. Recall before round 3 ----------
base = pd.read_parquet('pairs_final.parquet')
found_ids = found_from(base)
print("Recall before round 3: %.2f%%" % (100 * len(found_ids) / len(total_ids)))

# ---------- 3. Key builders ----------
cats = set()
for d in (df1, df2, df3):
    cats |= set(d['country'].astype(str).unique())
cmap = {c: i for i, c in enumerate(sorted(cats))}

FILLER = r'\b(?:plot|no|door|flat|house|shop|room|hno|apt|ste|unit|fl)\b'

def clean_addr(a):
    a = a.str.replace(r'(?<=[^\W\d_])(?=\d)', ' ', regex=True)  # no223 -> no 223
    a = a.str.replace(FILLER, ' ', regex=True)                   # drop plot/no/door/...
    a = a.str.replace(r'\b0+(?=\d)', '', regex=True)             # 00223 -> 223
    return a.str.replace(r'\s+', ' ', regex=True).str.strip()

def uniq_nums(x):
    s = set(x)
    return '|'.join(sorted(s)) if len(s) >= 2 else ''

KEYS = ['a_first2', 'a_nums', 'n_pre_num', 'n_suf_num', 'a_edges', 'n_init3']

def key_strings(d):
    addr = clean_addr(d['normalized_address'])
    name = d['normalized_name']
    first_num = addr.str.extract(r'(\d+)', expand=False).fillna('')
    first1 = addr.str.extract(r'^(\S+)', expand=False).fillna('')
    first2 = addr.str.extract(r'^(\S+(?: \S+)?)', expand=False).fillna('')
    last2 = addr.str.extract(r'(\S+(?: \S+)?)$', expand=False).fillna('')
    nums = addr.str.findall(r'\d+').map(uniq_nums)
    w = name.str.extract(r'^(\S{1,3})\S*\s+(\S{1,3})\S*(?:\s+(\S{1,3}))?', expand=True)
    init3 = (w[0].fillna('') + ' ' + w[1].fillna('') + ' ' + w[2].fillna('')).str.strip()
    init3 = init3.where(w[1].notna(), '')
    has_num = first_num.ne('') & name.ne('')
    return {
        'a_first2':  first2.where(first2.str.contains(' ', regex=False), ''),
        'a_nums':    nums,
        'n_pre_num': (name.str[:6] + '|' + first_num).where(has_num, ''),
        'n_suf_num': (name.str[-6:] + '|' + first_num).where(has_num, ''),
        'a_edges':   (first1 + '|' + last2).where(first1.ne('') & last2.str.contains(' ', regex=False), ''),
        'n_init3':   init3,
    }

def key_frames(d, id_name):
    ks = key_strings(d)
    ids = to_int_id(d['entity_id']).values
    ctry = d['country'].astype(str).map(cmap).astype('int16').values
    out = {}
    for k, s in ks.items():
        mask = s.ne('').values
        f = pd.DataFrame({id_name: ids[mask], 'country': ctry[mask]})
        f['key'] = pd.util.hash_pandas_object(s[mask], index=False).values
        out[k] = f
    return out

# ---------- 4. Right side (S2 + S3) tables, built in slices ----------
MAX_BLOCK = 50
SLICE = 1_000_000
parts = {k: [] for k in KEYS}
for d in (df2, df3):
    for s in range(0, len(d), SLICE):
        for k, f in key_frames(d.iloc[s:s + SLICE], 'id2').items():
            parts[k].append(f)
    print("right side keys built for one source")

right_tables = {}
for k in KEYS:
    r = pd.concat(parts[k], ignore_index=True)
    sizes = r.groupby(['country', 'key'])['id2'].transform('size')
    right_tables[k] = r[sizes <= MAX_BLOCK]
del parts, r, sizes

# ---------- 5. Chunked matching for df1, saved to disk ----------
os.makedirs('pairs_fuzzy3', exist_ok=True)
for f in os.listdir('pairs_fuzzy3'):
    os.remove(os.path.join('pairs_fuzzy3', f))

chunk = 100_000
total_new = 0
for i, start in enumerate(range(0, len(df1), chunk)):
    left = key_frames(df1.iloc[start:start + chunk], 'id1')
    outs = [
        left[k].merge(right_tables[k], on=['country', 'key'])[['id1', 'id2']]
        for k in KEYS
    ]
    out = pd.concat(outs).drop_duplicates().groupby('id1').head(100)
    out.to_parquet(f'pairs_fuzzy3/part_{i}.parquet')
    total_new += len(out)
    found_ids |= found_from(out)
    print(i, len(out))

print("\nRound 3 pairs saved (before dedupe vs old pairs):", total_new)
print("Blocking Recall AFTER round 3: %.2f%%" % (100 * len(found_ids) / len(total_ids)))

# ---------- 6. Merge everything into one file ----------
new_pairs = pd.concat(
    [pd.read_parquet(f'pairs_fuzzy3/{f}') for f in sorted(os.listdir('pairs_fuzzy3'))],
    ignore_index=True
)
all_pairs = pd.concat([base, new_pairs], ignore_index=True)
del base, new_pairs
all_pairs = all_pairs.drop_duplicates().sort_values(['id1', 'id2']).reset_index(drop=True)
all_pairs.to_parquet('pairs_final_v2.parquet')
print("Total pairs in pairs_final_v2.parquet:", len(all_pairs))
del all_pairs

# ---------- 7. WHY are the rest missed? (sample of 20k missed pairs) ----------
miss = gt[~gt['id1'].isin(found_ids)]
print("\nS1 rows with a true match but no candidate:", miss['id1'].nunique())
ms = miss.sample(min(20000, len(miss)), random_state=1).copy()
ms['s1'] = to_str_id(ms['id1'])
ms['s2'] = to_str_id(ms['id2'])

a = ms.merge(df1, left_on='s1', right_on='entity_id')
b = pd.concat([d[d['entity_id'].isin(ms['s2'])] for d in (df2, df3)])
b.columns = [c + '_true' for c in b.columns]
v = a.merge(b, left_on='s2', right_on='entity_id_true')

non_latin = lambda s: s.str.contains(r'[^\x00-\x7F]', regex=True)
cross_script = non_latin(v['normalized_name']) | non_latin(v['normalized_name_true'])
empty_addr = v['normalized_address'].eq('') | v['normalized_address_true'].eq('')
other = ~cross_script & ~empty_addr
print("Missed pairs (sample) where:")
print("  name is in a non-Latin script on either side: %.1f%%" % (100 * cross_script.mean()))
print("  address is empty on either side:              %.1f%%" % (100 * empty_addr.mean()))
print("  neither (real misses we could still fix):     %.1f%%" % (100 * other.mean()))

pd.set_option('display.max_colwidth', 45)
v.loc[other, ['normalized_name', 'normalized_name_true',
              'normalized_address', 'normalized_address_true',
              'country', 'country_true']].head(10)

Ground-truth rows with no valid match id (ignored): 123247
Recall before round 3: 96.27%
right side keys built for one source
right side keys built for one source
0 2163873
1 2170656
2 2175509
3 2161209
4 2174621
5 2158964
6 2169754
7 2170581
8 2176166
9 2166313
10 2180866
11 2167224
12 2164364
13 2162448
14 2168873
15 2163311
16 2158889
17 2168142
18 2167628
19 2153381
20 2170722
21 2167002
22 149510

Round 3 pairs saved (before dedupe vs old pairs): 47830006
Blocking Recall AFTER round 3: 98.00%
Total pairs in pairs_final_v2.parquet: 99977611

S1 rows with a true match but no candidate: 41723
Missed pairs (sample) where:
  name is in a non-Latin script on either side: 51.9%
  address is empty on either side:              5.8%
  neither (real misses we could still fix):     43.3%


,normalized_name,normalized_name_true,normalized_address,normalized_address_true,country,country_true
6,pediatric dental health,pediatric health dental,3322 waldens creek rd sevierville tn,332 waldens creek rd sevierville tn,us,us
8,oncology partners,oncology partners,719 5th st arkansas city ks,5th st arkanss city kansas,us,us
10,sunrise software,sunrise partners,uttar pradesh ghaziabad vaibhanv khand in...,b 2 101 vaibhanv khand indirapuram ghazia...,india,india
12,ortega national kgaa,ortega kgaa national,1110 5th st montevideo mn,5th st minnesota montevideo,us,us
13,zenith,zenith,garrett in 121 quincy st,12 quincy st garrett indiana,us,us
15,anchor nuclear p c,anchor nuclear,3207 westwood ave springfield mo,3071 westwood ave springfield missouri,us,us
16,urban producer,urban,opp mayank cinema rajasthan station rd ja...,malsisar house opp mayank cinema station ...,india,india
17,bison,bbirgnls,216 varner ave unit 1 orting wa,varner ave orting wa,us,us
19,laxmi exports,sri laxmi exports,humayun nagar hyderabad house no 12 2 717...,573 house no 12 2 717 1 78 saptagiri naga...,india,india
23,cardiology dynamic care,cardiologydynamiccare com,2301 deborah dr ny niskayuna,deborah dr schenectady new york,us,us


In [19]:
import os, re, time
import numpy as np
import pandas as pd
import regex
import lightgbm as lgb
from rapidfuzz import fuzz

# ==========================================
# ML STEP v2: labels + features + LightGBM + F0.5 threshold
# Needs: df1, df2, df3 (normalized), pairs_final_v2.parquet (or pairs_final.parquet)
# ==========================================

# ---------- 0. Settings ----------
TRAIN_MODS = (1, 2)      # train on S1 rows where id1 % 10 in these (~20% of rows)
NEG_FRAC = 0.2           # keep 20% of negatives in training
VAL_MOD = 50             # validate on S1 rows where id1 % 50 == 0 (~2% of rows)
CHUNK = 500_000
BETA = 0.5               # F_0.5: precision counts more than recall

def to_int_id(s):
    s = s.astype(str)
    return s.str[1:2].astype('int64') * 10**10 + s.str[3:].astype('int64')

# real non-Latin scripts only (Hindi, Telugu, ...), NOT accents like é or ç
NONLATIN = regex.compile(r'[\p{L}--\p{Latin}]', flags=regex.V1)

def nums(t):
    return {x.lstrip('0') for x in re.findall(r'\d+', t)}

# ---------- 1. Ground truth ----------
gt = pd.read_csv(
    './student_resource/student_resource/dataset/train/train_ground_truth.tsv',
    sep='\t'
)[['source1_entity_id', 'matched_entity_ids']]
gt['id2'] = gt['matched_entity_ids'].astype(str).str.split(',')
gt = gt.explode('id2')
gt['id2'] = gt['id2'].str.strip()
ok = (gt['id2'].str.match(r'^S\d-\d+$')
      & gt['source1_entity_id'].astype(str).str.match(r'^S\d-\d+$'))
gt = gt[ok].copy()
gt['id1'] = to_int_id(gt['source1_entity_id'])
gt['id2'] = to_int_id(gt['id2'])
gt = gt[['id1', 'id2']].drop_duplicates()
gt['label'] = 1
print("Ground-truth pairs:", len(gt))

# does each S2/S3 record belong to at most one S1 entity?
multi = gt.groupby('id2')['id1'].nunique()
print("S2/S3 records matched to MORE than one S1 entity: %.2f%%" % (100 * (multi > 1).mean()))

# ---------- 2. Load pairs, counts, split by id1 ----------
path = 'pairs_final_v2.parquet' if os.path.exists('pairs_final_v2.parquet') else 'pairs_final.parquet'
print("Using", path)
pairs = pd.read_parquet(path)
cnt1 = pairs['id1'].value_counts()      # candidates per S1 row
cnt2 = pairs['id2'].value_counts()      # S1 rows competing for each S2/S3 row

is_train = (pairs['id1'] % 10).isin(TRAIN_MODS)
is_val = (pairs['id1'] % VAL_MOD) == 0

train = pairs[is_train].merge(gt, on=['id1', 'id2'], how='left')
train['label'] = train['label'].fillna(0).astype('int8')
val = pairs[is_val].merge(gt, on=['id1', 'id2'], how='left')
val['label'] = val['label'].fillna(0).astype('int8')
del pairs

neg = train[train['label'] == 0].sample(frac=NEG_FRAC, random_state=0)
train = pd.concat([train[train['label'] == 1], neg]).sample(frac=1, random_state=0).reset_index(drop=True)
del neg
print("Train rows:", len(train), "| positives:", int(train['label'].sum()))
print("Val rows:", len(val), "| positives:", int(val['label'].sum()))

# all true pairs for val S1 rows, including ones blocking never found
total_pos_val = int(gt[(gt['id1'] % VAL_MOD) == 0]['label'].sum())
print("Val true pairs (incl. missed by blocking):", total_pos_val)

# ---------- 3. Text lookups ----------
t1 = pd.DataFrame({'name': df1['normalized_name'].values,
                   'addr': df1['normalized_address'].values},
                  index=to_int_id(df1['entity_id']).values)
d23 = pd.concat([df2, df3], ignore_index=True)
t2 = pd.DataFrame({'name': d23['normalized_name'].values,
                   'addr': d23['normalized_address'].values},
                  index=to_int_id(d23['entity_id']).values)
del d23

# ---------- 4. Feature builder ----------
def build_features(p):
    out = []
    t0 = time.time()
    for s in range(0, len(p), CHUNK):
        c = p.iloc[s:s + CHUNK]
        i1 = t1.index.get_indexer(c['id1'].values)
        i2 = t2.index.get_indexer(c['id2'].values)
        n1 = t1['name'].values[i1]; n2 = t2['name'].values[i2]
        a1 = t1['addr'].values[i1]; a2 = t2['addr'].values[i2]

        f = pd.DataFrame(index=c.index)
        f['name_ratio'] = [fuzz.ratio(x, y) for x, y in zip(n1, n2)]
        f['name_tsort'] = [fuzz.token_sort_ratio(x, y) for x, y in zip(n1, n2)]
        f['name_tset'] = [fuzz.token_set_ratio(x, y) for x, y in zip(n1, n2)]
        f['name_partial'] = [fuzz.partial_ratio(x, y) for x, y in zip(n1, n2)]
        f['addr_ratio'] = [fuzz.ratio(x, y) for x, y in zip(a1, a2)]
        f['addr_tsort'] = [fuzz.token_sort_ratio(x, y) for x, y in zip(a1, a2)]
        f['addr_tset'] = [fuzz.token_set_ratio(x, y) for x, y in zip(a1, a2)]
        f['name_exact'] = [int(x == y and x != '') for x, y in zip(n1, n2)]
        f['addr_exact'] = [int(x == y and x != '') for x, y in zip(a1, a2)]
        f['name_first_eq'] = [int(x.split()[:1] == y.split()[:1] and x != '') for x, y in zip(n1, n2)]

        s1 = [nums(x) for x in a1]; s2 = [nums(x) for x in a2]
        inter = np.array([len(x & y) for x, y in zip(s1, s2)])
        union = np.array([len(x | y) for x, y in zip(s1, s2)])
        f['num_inter'] = inter
        f['num_jacc'] = np.where(union > 0, inter / np.maximum(union, 1), 0)
        f['num_conflict'] = [int(len(x) > 0 and len(y) > 0 and len(x & y) == 0) for x, y in zip(s1, s2)]

        f['addr1_empty'] = [int(x == '') for x in a1]
        f['addr2_empty'] = [int(x == '') for x in a2]
        nl1 = np.array([int(bool(NONLATIN.search(x))) for x in n1])
        nl2 = np.array([int(bool(NONLATIN.search(x))) for x in n2])
        f['name1_nonlatin'] = nl1
        f['name2_nonlatin'] = nl2
        f['script_mismatch'] = (nl1 != nl2).astype(int)
        f['name_len_diff'] = [abs(len(x) - len(y)) for x, y in zip(n1, n2)]
        f['addr_len_diff'] = [abs(len(x) - len(y)) for x, y in zip(a1, a2)]
        f['name_tok_diff'] = [abs(len(x.split()) - len(y.split())) for x, y in zip(n1, n2)]
        f['cnt_id1'] = cnt1.reindex(c['id1'].values).values
        f['cnt_id2'] = cnt2.reindex(c['id2'].values).values
        out.append(f.astype('float32'))
        done = min(s + CHUNK, len(p))
        print("  features: %d / %d  (%.0f pairs/sec)" % (done, len(p), done / (time.time() - t0)))
    return pd.concat(out)

print("Building train features...")
Xtr = build_features(train); ytr = train['label'].values
print("Building val features...")
Xva = build_features(val); yva = val['label'].values

# ---------- 5. Train LightGBM ----------
params = dict(
    objective='binary', metric='average_precision',
    learning_rate=0.05, num_leaves=63, min_data_in_leaf=50,
    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
    verbose=-1, seed=0,
)
dtr = lgb.Dataset(Xtr, ytr)
dva = lgb.Dataset(Xva, yva, reference=dtr)
model = lgb.train(
    params, dtr, num_boost_round=1500, valid_sets=[dva],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

# ---------- 6. Threshold search on validation (maximize F_0.5) ----------
prob = model.predict(Xva, num_iteration=model.best_iteration)

def fbeta(prec, rec, beta):
    b2 = beta ** 2
    return (1 + b2) * prec * rec / max(b2 * prec + rec, 1e-9)

best = (0, 0.5)
print("\nthr   precision  recall   F0.5    F1   (recall counts matches blocking missed)")
for thr in np.arange(0.10, 0.99, 0.025):
    pred = prob >= thr
    tp = int((pred & (yva == 1)).sum())
    fp = int(pred.sum()) - tp
    prec = tp / max(tp + fp, 1)
    rec = tp / max(total_pos_val, 1)
    f05 = fbeta(prec, rec, BETA)
    f1 = fbeta(prec, rec, 1.0)
    print("%.3f  %.4f     %.4f   %.4f  %.4f" % (thr, prec, rec, f05, f1))
    if f05 > best[0]:
        best = (f05, thr)
print("\nBest F0.5 %.4f at threshold %.3f" % best)

imp = pd.Series(model.feature_importance('gain'), index=Xtr.columns).sort_values(ascending=False)
print("\nTop features by gain:\n", imp.head(12))

# ---------- 7. Save ----------
model.save_model('lgb_matcher.txt')
with open('threshold.txt', 'w') as fh:
    fh.write(str(best[1]))
print("Saved lgb_matcher.txt and threshold.txt")

Ground-truth pairs: 7638365
S2/S3 records matched to MORE than one S1 entity: 0.00%
Using pairs_final_v2.parquet
Train rows: 5065473 | positives: 1330810
Val rows: 2014881 | positives: 133587
Val true pairs (incl. missed by blocking): 153667
Building train features...
  features: 500000 / 5065473  (22254 pairs/sec)
  features: 1000000 / 5065473  (24413 pairs/sec)
  features: 1500000 / 5065473  (25019 pairs/sec)
  features: 2000000 / 5065473  (25459 pairs/sec)
  features: 2500000 / 5065473  (25671 pairs/sec)
  features: 3000000 / 5065473  (25943 pairs/sec)
  features: 3500000 / 5065473  (26085 pairs/sec)
  features: 4000000 / 5065473  (26169 pairs/sec)
  features: 4500000 / 5065473  (26265 pairs/sec)
  features: 5000000 / 5065473  (26313 pairs/sec)
  features: 5065473 / 5065473  (26303 pairs/sec)
Building val features...
  features: 500000 / 2014881  (28577 pairs/sec)
  features: 1000000 / 2014881  (28010 pairs/sec)
  features: 1500000 / 2014881  (27731 pairs/sec)
  features: 2000000 / 

In [20]:
import os, re, time, csv, gc, unicodedata
import numpy as np
import pandas as pd
import regex
import lightgbm as lgb
from rapidfuzz import fuzz

# ==========================================
# FINAL PIPELINE: test files -> candidate_pairs.tsv + matching_results.tsv
# ==========================================

# ---------- 0. Settings ----------
TEST_DIR = './student_resource/student_resource/dataset/test/'
OUT_DIR = './student_resource/student_resource/output/'
ONE_S1_PER_RECORD = True   # set False if the training check printed a clearly >0% "more than one S1 entity"
LEFT_CHUNK = 100_000
SLICE = 1_000_000
FEAT_CHUNK = 500_000

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs('test_pairs', exist_ok=True)
for f in os.listdir('test_pairs'):
    os.remove(os.path.join('test_pairs', f))

# free training memory (reload from the *_norm.parquet files if you need them again)
for v in ['Xtr', 'Xva', 'train', 'val', 'dtr', 'dva', 'prob', 't1', 't2',
          'cnt1', 'cnt2', 'gt', 'multi', 'df1', 'df2', 'df3', 'pairs']:
    globals().pop(v, None)
gc.collect()

model = lgb.Booster(model_file='lgb_matcher.txt')
THR = float(open('threshold.txt').read())
print("Threshold:", THR)

# ---------- 1. Load + normalize test data (with accent stripping for Latin letters only) ----------
LATIN_MARKS = regex.compile(r'(?<=\p{Latin})\p{M}+')   # Hindi matras are NOT touched

def strip_latin_accents(lst):
    out = []
    for t in lst:
        t = unicodedata.normalize('NFD', str(t))
        t = LATIN_MARKS.sub('', t)
        out.append(unicodedata.normalize('NFC', t))
    return out

cols = ['entity_id', 'business_name', 'business_address', 'country']

def prep(fname):
    d = pd.read_csv(TEST_DIR + fname, sep='\t', usecols=cols, engine='pyarrow')
    d['normalized_name'] = norm_col(d['business_name'], lambda l: normalize_name(strip_latin_accents(l)))
    d['normalized_address'] = norm_col(d['business_address'], lambda l: normalize_address(strip_latin_accents(l)))
    d['country'] = d['country'].astype(str).str.lower().str.strip()
    return d[['entity_id', 'country', 'normalized_name', 'normalized_address']]

s1 = prep('test_source1.tsv')
s2 = prep('test_source2.tsv')
s3 = prep('test_source3.tsv')
d23 = pd.concat([s2, s3], ignore_index=True)   # id2 = position in this table
print("Rows: S1 %d | S2 %d | S3 %d" % (len(s1), len(s2), len(s3)))
print("Countries in S1:", s1['country'].value_counts().to_dict())

ids1 = s1['entity_id'].astype(str).values
ids23 = d23['entity_id'].astype(str).values
nm1, ad1 = s1['normalized_name'].values, s1['normalized_address'].values
nm2, ad2 = d23['normalized_name'].values, d23['normalized_address'].values

cats = sorted(set(s1['country']) | set(d23['country']))
cmap = {c: i for i, c in enumerate(cats)}

# ---------- 2. Blocking keys (same keys as exact + rounds 1, 2, 3) ----------
FILLER = r'\b(?:plot|no|door|flat|house|shop|room|hno|apt|ste|unit|fl)\b'

def clean_addr(a):
    a = a.str.replace(r'(?<=[^\W\d_])(?=\d)', ' ', regex=True)
    a = a.str.replace(FILLER, ' ', regex=True)
    a = a.str.replace(r'\b0+(?=\d)', '', regex=True)
    return a.str.replace(r'\s+', ' ', regex=True).str.strip()

def uniq_nums(x):
    s = set(x)
    return '|'.join(sorted(s)) if len(s) >= 2 else ''

def sort_words(lst):
    return [' '.join(sorted(s.split())) for s in lst]

def all_keys(d):
    name = d['normalized_name']
    addr0 = d['normalized_address']
    ks = {}
    # exact
    ks['x_name'] = name
    ks['x_addr'] = addr0
    # round 1
    ks['r1_name'] = name.str[:8]
    ks['r1_addr'] = addr0.str[:12]
    # round 2
    a2 = addr0.str.replace(r'\b0+(?=\d)', '', regex=True)
    sname = norm_col(name, sort_words)
    fn2 = a2.str.extract(r'(\d+)', expand=False).fillna('')
    ex = a2.str.extract(r'\b(\d+)\s+([^\W\d_]+)', expand=True)
    ks['r2_addr'] = a2.str[:12]
    ks['r2_sortname'] = (sname + '|' + fn2).where(sname.ne('') & fn2.ne(''), '')
    ks['r2_numtok'] = (ex[0] + ' ' + ex[1]).fillna('')
    # round 3
    a3 = clean_addr(addr0)
    fn3 = a3.str.extract(r'(\d+)', expand=False).fillna('')
    first1 = a3.str.extract(r'^(\S+)', expand=False).fillna('')
    first2 = a3.str.extract(r'^(\S+(?: \S+)?)', expand=False).fillna('')
    last2 = a3.str.extract(r'(\S+(?: \S+)?)$', expand=False).fillna('')
    nums3 = a3.str.findall(r'\d+').map(uniq_nums)
    w = name.str.extract(r'^(\S{1,3})\S*\s+(\S{1,3})\S*(?:\s+(\S{1,3}))?', expand=True)
    init3 = (w[0].fillna('') + ' ' + w[1].fillna('') + ' ' + w[2].fillna('')).str.strip()
    init3 = init3.where(w[1].notna(), '')
    has_num = fn3.ne('') & name.ne('')
    ks['r3_first2'] = first2.where(first2.str.contains(' ', regex=False), '')
    ks['r3_nums'] = nums3
    ks['r3_name_pre_num'] = (name.str[:6] + '|' + fn3).where(has_num, '')
    ks['r3_name_suf_num'] = (name.str[-6:] + '|' + fn3).where(has_num, '')
    ks['r3_edges'] = (first1 + '|' + last2).where(first1.ne('') & last2.str.contains(' ', regex=False), '')
    ks['r3_init3'] = init3
    return ks

# family -> (keys, max block size on the S2/S3 side, per-row cap)
FAMILIES = {
    'exact': (['x_name', 'x_addr'], 200, None),
    'r1': (['r1_name', 'r1_addr'], 50, 100),
    'r2': (['r2_addr', 'r2_sortname', 'r2_numtok'], 50, 100),
    'r3': (['r3_first2', 'r3_nums', 'r3_name_pre_num', 'r3_name_suf_num', 'r3_edges', 'r3_init3'], 50, 100),
}
ALL_KEYS = [k for ks, _, _ in FAMILIES.values() for k in ks]
MAXB = {k: mb for ks, mb, _ in FAMILIES.values() for k in ks}

def key_frames(d, ids, id_name):
    ks = all_keys(d)
    ctry = d['country'].map(cmap).astype('int16').values
    out = {}
    for k in ALL_KEYS:
        s = ks[k]
        mask = s.ne('').values
        f = pd.DataFrame({id_name: ids[mask], 'country': ctry[mask]})
        f['key'] = pd.util.hash_pandas_object(s[mask], index=False).values
        out[k] = f
    return out

def num_frames(d, ids, id_name):
    a = d['normalized_address'].str.replace(r'\b0+(?=\d)', '', regex=True)
    t = pd.DataFrame({id_name: ids,
                      'country': d['country'].map(cmap).astype('int16').values,
                      'n': a.str.findall(r'\d+').values}).explode('n')
    t = t[t['n'].notna()]
    t['key'] = pd.util.hash_pandas_object(t['n'], index=False).values
    return t[[id_name, 'country', 'key']]

def merge_pairs(l, r):
    return l.merge(r, on=['country', 'key'])[['id1', 'id2']]

# ---------- 3. Right-side (S2 + S3) tables, built once ----------
print("Building right-side key tables...")
parts = {k: [] for k in ALL_KEYS}
num_parts = []
for s in range(0, len(d23), SLICE):
    sl = d23.iloc[s:s + SLICE]
    ids = np.arange(s, s + len(sl))
    for k, f in key_frames(sl, ids, 'id2').items():
        parts[k].append(f)
    num_parts.append(num_frames(sl, ids, 'id2'))
    print("  slice done:", s)

right = {}
for k in ALL_KEYS:
    r = pd.concat(parts[k], ignore_index=True)
    sizes = r.groupby(['country', 'key'])['id2'].transform('size')
    right[k] = r[sizes <= MAXB[k]]
    parts[k] = None
rnum = pd.concat(num_parts, ignore_index=True)
rnum = rnum[rnum.groupby(['country', 'key'])['id2'].transform('size') <= 100]
del parts, num_parts, r, sizes

# ---------- 4. Candidate generation (chunked) -> candidate_pairs.tsv + test_pairs/ ----------
cnt1 = np.zeros(len(s1), dtype=np.int64)     # candidates per S1 row
cnt2 = np.zeros(len(d23), dtype=np.int64)    # S1 rows competing for each S2/S3 row

with open(OUT_DIR + 'candidate_pairs.tsv', 'w', encoding='utf-8', newline='') as fh:
    fh.write('source1_entity_id\tcandidate_entity_ids\n')
    for i, start in enumerate(range(0, len(s1), LEFT_CHUNK)):
        part = s1.iloc[start:start + LEFT_CHUNK]
        ids = np.arange(start, start + len(part))
        left = key_frames(part, ids, 'id1')

        fam_out = []
        for fam, (keys, mb, cap) in FAMILIES.items():
            o = pd.concat([merge_pairs(left[k], right[k]) for k in keys]).drop_duplicates()
            if cap:
                o = o.groupby('id1').head(cap)
            fam_out.append(o)
        out = pd.concat(fam_out).drop_duplicates()

        # fallback: address numbers, only for rows with no candidates
        missing = np.setdiff1d(ids, out['id1'].unique())
        if len(missing):
            sub = part.iloc[missing - start]
            fb = merge_pairs(num_frames(sub, missing, 'id1'), rnum).drop_duplicates().groupby('id1').head(100)
            out = pd.concat([out, fb]).drop_duplicates()

        out = out[['id1', 'id2']].reset_index(drop=True)
        out.to_parquet(f'test_pairs/part_{i}.parquet')
        cnt1 += np.bincount(out['id1'].values, minlength=len(s1))
        cnt2 += np.bincount(out['id2'].values, minlength=len(d23))

        # write candidate lines for every S1 row in this chunk (empty when none)
        strs = pd.Series(ids23[out['id2'].values])
        grouped = strs.groupby(out['id1'].values).agg(','.join)
        lines = pd.Series('', index=ids)
        lines.loc[grouped.index] = grouped.values
        cand_str = lines.values
        fh.write(''.join(ids1[j] + '\t' + cand_str[j - start] + '\n' for j in ids))
        print("chunk", i, "pairs:", len(out))

print("S1 rows with at least one candidate: %d of %d" % ((cnt1 > 0).sum(), len(s1)))
print("Average candidates per S1 row: %.1f" % cnt1.mean())

# ---------- 5. Features + scoring (same features/order as training) ----------
NONLATIN = regex.compile(r'[\p{L}--\p{Latin}]', flags=regex.V1)

def nums(t):
    return {x.lstrip('0') for x in re.findall(r'\d+', t)}

def featurize(p):
    out = []
    t0 = time.time()
    for s in range(0, len(p), FEAT_CHUNK):
        c = p.iloc[s:s + FEAT_CHUNK]
        i1 = c['id1'].values
        i2 = c['id2'].values
        n1, n2, a1, a2 = nm1[i1], nm2[i2], ad1[i1], ad2[i2]

        f = pd.DataFrame(index=c.index)
        f['name_ratio'] = [fuzz.ratio(x, y) for x, y in zip(n1, n2)]
        f['name_tsort'] = [fuzz.token_sort_ratio(x, y) for x, y in zip(n1, n2)]
        f['name_tset'] = [fuzz.token_set_ratio(x, y) for x, y in zip(n1, n2)]
        f['name_partial'] = [fuzz.partial_ratio(x, y) for x, y in zip(n1, n2)]
        f['addr_ratio'] = [fuzz.ratio(x, y) for x, y in zip(a1, a2)]
        f['addr_tsort'] = [fuzz.token_sort_ratio(x, y) for x, y in zip(a1, a2)]
        f['addr_tset'] = [fuzz.token_set_ratio(x, y) for x, y in zip(a1, a2)]
        f['name_exact'] = [int(x == y and x != '') for x, y in zip(n1, n2)]
        f['addr_exact'] = [int(x == y and x != '') for x, y in zip(a1, a2)]
        f['name_first_eq'] = [int(x.split()[:1] == y.split()[:1] and x != '') for x, y in zip(n1, n2)]

        sa = [nums(x) for x in a1]
        sb = [nums(x) for x in a2]
        inter = np.array([len(x & y) for x, y in zip(sa, sb)])
        union = np.array([len(x | y) for x, y in zip(sa, sb)])
        f['num_inter'] = inter
        f['num_jacc'] = np.where(union > 0, inter / np.maximum(union, 1), 0)
        f['num_conflict'] = [int(len(x) > 0 and len(y) > 0 and len(x & y) == 0) for x, y in zip(sa, sb)]

        f['addr1_empty'] = [int(x == '') for x in a1]
        f['addr2_empty'] = [int(x == '') for x in a2]
        nl1 = np.array([int(bool(NONLATIN.search(x))) for x in n1])
        nl2 = np.array([int(bool(NONLATIN.search(x))) for x in n2])
        f['name1_nonlatin'] = nl1
        f['name2_nonlatin'] = nl2
        f['script_mismatch'] = (nl1 != nl2).astype(int)
        f['name_len_diff'] = [abs(len(x) - len(y)) for x, y in zip(n1, n2)]
        f['addr_len_diff'] = [abs(len(x) - len(y)) for x, y in zip(a1, a2)]
        f['name_tok_diff'] = [abs(len(x.split()) - len(y.split())) for x, y in zip(n1, n2)]
        f['cnt_id1'] = cnt1[i1]
        f['cnt_id2'] = cnt2[i2]
        out.append(f.astype('float32'))
        done = min(s + FEAT_CHUNK, len(p))
        print("    features %d / %d (%.0f pairs/sec)" % (done, len(p), done / (time.time() - t0)))
    return pd.concat(out)

kept = []
total = 0
for fpart in sorted(os.listdir('test_pairs')):
    p = pd.read_parquet('test_pairs/' + fpart)
    X = featurize(p)[model.feature_name()]
    pr = model.predict(X)
    keep = pr >= THR
    kept.append(pd.DataFrame({'id1': p['id1'].values[keep],
                              'id2': p['id2'].values[keep],
                              'prob': pr[keep]}))
    total += len(p)
    print(fpart, "scored:", len(p), "| above threshold:", int(keep.sum()))

# ---------- 6. Final matches -> matching_results.tsv ----------
matches = pd.concat(kept, ignore_index=True)
print("\nTotal pairs scored:", total)
print("Pairs above threshold:", len(matches))
if ONE_S1_PER_RECORD:
    matches = matches.sort_values('prob', ascending=False).drop_duplicates('id2')
    print("After one-S1-per-record rule:", len(matches))

matches['m'] = ids23[matches['id2'].values]
g = matches.groupby('id1')['m'].agg(','.join)
res = pd.Series('', index=np.arange(len(s1)))
res.loc[g.index] = g.values

result = pd.DataFrame({'source1_entity_id': ids1, 'matched_entity_ids': res.values})
result.to_csv(OUT_DIR + 'matching_results.tsv', sep='\t', index=False,
              quoting=csv.QUOTE_NONE, lineterminator='\n')

print("S1 entities with at least one match: %d of %d (%.1f%%)" % (
    (result['matched_entity_ids'] != '').sum(), len(result),
    100 * (result['matched_entity_ids'] != '').mean()))
print("\nDone. Validate with (run from the student_resource folder):")
print("python3 utils/validate_submission.py --matching output/matching_results.tsv "
      "--candidate output/candidate_pairs.tsv --test-dir dataset/test")

Threshold: 0.9249999999999998
Rows: S1 1732544 | S2 4887273 | S3 5082316
Countries in S1: {'india': 809986, 'us': 663106, 'france': 259452}
Building right-side key tables...
  slice done: 0
  slice done: 1000000
  slice done: 2000000
  slice done: 3000000
  slice done: 4000000
  slice done: 5000000
  slice done: 6000000
  slice done: 7000000
  slice done: 8000000
  slice done: 9000000
chunk 0 pairs: 4388202
chunk 1 pairs: 4375587
chunk 2 pairs: 4380931
chunk 3 pairs: 4353019
chunk 4 pairs: 4400721
chunk 5 pairs: 4380934
chunk 6 pairs: 4362747
chunk 7 pairs: 4379518
chunk 8 pairs: 4372992
chunk 9 pairs: 4393857
chunk 10 pairs: 4387905
chunk 11 pairs: 4349864
chunk 12 pairs: 4390290
chunk 13 pairs: 4345113
chunk 14 pairs: 4379652
chunk 15 pairs: 4376360
chunk 16 pairs: 4393917
chunk 17 pairs: 1428174
S1 rows with at least one candidate: 1730608 of 1732544
Average candidates per S1 row: 43.8
    features 500000 / 4388202 (28751 pairs/sec)
    features 1000000 / 4388202 (28783 pairs/sec)
 

In [21]:
import subprocess, sys
import pandas as pd

BASE = './student_resource/student_resource/'

# ---------- 1. Official validator ----------
r = subprocess.run(
    [sys.executable, 'utils/validate_submission.py',
     '--matching', 'output/matching_results.tsv',
     '--candidate', 'output/candidate_pairs.tsv',
     '--test-dir', 'dataset/test'],
    cwd=BASE, capture_output=True, text=True
)
print(r.stdout)
print(r.stderr)
print("exit code:", r.returncode, "(0 = PASS)")

# ---------- 2. Sanity check: match rate per country vs training ----------
res = pd.read_csv(BASE + 'output/matching_results.tsv', sep='\t', dtype=str, keep_default_na=False)
s1 = pd.read_csv(BASE + 'dataset/test/test_source1.tsv', sep='\t',
                 usecols=['entity_id', 'country'], dtype=str)
m = s1.merge(res, left_on='entity_id', right_on='source1_entity_id')
m['has_match'] = m['matched_entity_ids'].ne('')
print("\nTest: share of S1 entities with a match, by country")
print(m.groupby(m['country'].str.lower().str.strip())['has_match'].mean().round(3))

gt = pd.read_csv(BASE + 'dataset/train/train_ground_truth.tsv', sep='\t', dtype=str, keep_default_na=False)
print("\nTraining: share of S1 entities with a match: %.1f%%" % (100 * gt['matched_entity_ids'].ne('').mean()))

ML Challenge 2026 â€” submission validator
  test dir: dataset/test
  required S1 entities: 1732544
  matching_results.tsv: 1732544 rows (154624 empty, 1577920 non-empty).
  candidate_pairs.tsv: 1732544 rows (1936 empty, 1730608 non-empty).

PASS â€” no blocking issues found. Safe to submit.


exit code: 0 (0 = PASS)

Test: share of S1 entities with a match, by country
country
france    0.937
india     0.894
us        0.921
Name: has_match, dtype: float64

Training: share of S1 entities with a match: 94.4%
